# Évaluation des légendes et jeu de données pour une LoRA

*Projet 7 : VLMs légers pour légender ou juger des images.*

Mesure de la qualité des légendes (BLEU, ROUGE-L, CIDEr, CLIPScore), graphiques de comparaison et export d'un jeu de données image + légende pour entraîner une LoRA.

GPU requis dans Colab : *Exécution → Modifier le type d'exécution → T4 GPU*. Exécute les cellules dans l'ordre ; ce qui a déjà été calculé est repris depuis le Drive partagé.

## Mise en place (commune aux trois notebooks)

> **Dossier partagé** : Wafae crée `projet7_vlm` dans son Google Drive et le partage avec Meriem et Fatima (en éditrices). Meriem et Fatima l'ajoutent à *Mon Drive* : clic droit → *Organiser* → *Ajouter un raccourci à Drive*. Ainsi `MyDrive/projet7_vlm` pointe vers le même dossier pour les trois.
>
> **Ne pas modifier** `SEED`, `N_IMAGES`, `MAX_SIDE` sans accord de l'équipe : sinon les images ne sont plus les mêmes.

In [ ]:
!pip install -q -U "transformers>=4.49" accelerate bitsandbytes datasets pycocoevalcap gradio num2words scipy

In [ ]:
import os, gc, json, re, time, random, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

assert torch.cuda.is_available(), "Aucun GPU détecté : active un GPU (Exécution → Modifier le type d'exécution)."
print("GPU :", torch.cuda.get_device_name(0))
print("Mémoire GPU totale : %.1f Go" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

# bfloat16 seulement sur GPU Ampere ou plus récent (A100, L4...). Sur T4 → float16.
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print("Type des poids en 16 bits :", DTYPE)

# ------------------------- CONFIGURATION -------------------------
SEED = 42
N_IMAGES = 300            # taille du sous-ensemble évalué (le sujet conseille 500 à 2000 ; 300 ≈ 1 h sur T4)
MAX_SIDE = 512            # redimensionnement des images (côté le plus long, en pixels)
MAX_NEW_TOKENS = 40       # longueur max d'une légende

MODELS = {
    "Qwen2-VL-2B": "Qwen/Qwen2-VL-2B-Instruct",
    "SmolVLM-2B":  "HuggingFaceTB/SmolVLM-Instruct",
}
PRECISIONS = ["16bit", "8bit", "4bit"]

CAPTION_PROMPT = "Describe this image in one short sentence."

# Option : sauvegarder sur Google Drive pour ne rien perdre en cas de déconnexion
USE_DRIVE = True   # dossier partagé de l'équipe : MyDrive/projet7_vlm
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/projet7_vlm")
else:
    ROOT = Path("/content/projet7_vlm") if Path("/content").exists() else Path("projet7_vlm")

DATA_DIR = ROOT / "data"
IMG_DIR = DATA_DIR / "images"
RES_DIR = ROOT / "results"
for d in (IMG_DIR, RES_DIR):
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Dossier de travail :", ROOT)

## 2. Préparation des données — Flickr8k

Flickr8k contient ~8 000 images, chacune avec **5 légendes humaines**. On prend un sous-ensemble aléatoire (graine fixe) de la partie *test*, on **redimensionne** les images (côté max = `MAX_SIDE`) et on les enregistre sur disque : **les mêmes images** serviront à toutes les configurations (16 / 8 / 4 bits), ce qui rend la comparaison équitable.

In [ ]:
from datasets import load_dataset

def prepare_image(img, max_side=MAX_SIDE):
    # Convertit en RGB et réduit l'image pour que son plus grand côté fasse max_side pixels.
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side), Image.BICUBIC)
    return img

META_PATH = DATA_DIR / "subset.json"

if META_PATH.exists():
    subset = json.loads(META_PATH.read_text())
    print(f"Sous-ensemble déjà préparé : {len(subset)} images")
else:
    ds = load_dataset("jxie/flickr8k", split="test")          # 1000 images, colonnes image + caption_0..4
    caption_cols = sorted(c for c in ds.column_names if c.startswith("caption"))
    idx = sorted(random.Random(SEED).sample(range(len(ds)), min(N_IMAGES, len(ds))))
    subset = []
    for i in tqdm(idx, desc="Préparation des images"):
        ex = ds[i]
        img_id = f"img_{i:04d}"
        prepare_image(ex["image"]).save(IMG_DIR / f"{img_id}.jpg", quality=95)
        subset.append({"id": img_id, "refs": [ex[c].strip() for c in caption_cols]})
    META_PATH.write_text(json.dumps(subset, indent=1, ensure_ascii=False))
    print(f"{len(subset)} images préparées dans {IMG_DIR}")

REFS = {s["id"]: s["refs"] for s in subset}
IDS = [s["id"] for s in subset]

def load_img(img_id):
    return Image.open(IMG_DIR / f"{img_id}.jpg").convert("RGB")

## 5. Évaluation

**Métriques par rapport aux 5 légendes humaines** :
- **BLEU-1 / BLEU-4** : recouvrement de mots et de groupes de 4 mots ;
- **ROUGE-L** : plus longue sous-séquence commune ;
- **CIDEr** : la métrique de référence en légendage (pondère les mots informatifs) ;
- **CLIPScore** : accord image ↔ légende mesuré par CLIP, **sans** référence humaine.

**Fidélité au 16 bits** : à quel point la version quantifiée produit la même légende que la version 16 bits (BLEU-4 contre la légende 16 bits, % de légendes identiques).

> Remarque : les VLMs modernes écrivent des phrases plus longues et détaillées que les légendes de Flickr8k, donc BLEU/CIDEr sont bas en absolu. Ce qui compte ici, c'est la **comparaison** entre modèles et précisions. Le CLIPScore complète en évaluant la justesse visuelle.

In [ ]:
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

def tok(s):
    t = " ".join(re.findall(r"[a-z0-9]+", str(s).lower()))
    return t if t else "empty"

def text_metrics(preds, refs):
    # preds : {id: légende}, refs : {id: [légendes de référence]}
    gts = {k: [tok(r) for r in refs[k]] for k in preds}
    res = {k: [tok(preds[k])] for k in preds}
    bleu, _ = Bleu(4).compute_score(gts, res, verbose=0)
    rouge, _ = Rouge().compute_score(gts, res)
    cider, _ = Cider().compute_score(gts, res)
    return {"BLEU-1": bleu[0], "BLEU-4": bleu[3], "ROUGE-L": rouge, "CIDEr": cider}

In [ ]:
# CLIPScore (Hessel et al., 2021) : 2.5 * max(cos(image, texte), 0)
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").half().cuda().eval()

def _as_tensor(e):
    # Selon la version de transformers, get_*_features renvoie un tenseur ou un objet de sortie
    if torch.is_tensor(e):
        return e
    for attr in ("image_embeds", "text_embeds", "pooler_output"):
        if getattr(e, attr, None) is not None:
            return getattr(e, attr)
    raise TypeError(type(e))
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

@torch.inference_mode()
def clip_image_emb(ids, bs=32):
    embs = []
    for i in range(0, len(ids), bs):
        imgs = [load_img(x) for x in ids[i:i + bs]]
        px = clip_proc(images=imgs, return_tensors="pt")["pixel_values"].cuda().half()
        e = clip_model.get_image_features(pixel_values=px)
        embs.append(torch.nn.functional.normalize(_as_tensor(e).float(), dim=-1).cpu())
    return dict(zip(ids, torch.cat(embs)))

@torch.inference_mode()
def clip_text_emb(texts, bs=64):
    embs = []
    for i in range(0, len(texts), bs):
        t = clip_proc(text=list(texts[i:i + bs]), return_tensors="pt", padding=True,
                      truncation=True, max_length=77)
        e = clip_model.get_text_features(input_ids=t["input_ids"].cuda(), attention_mask=t["attention_mask"].cuda())
        embs.append(torch.nn.functional.normalize(_as_tensor(e).float(), dim=-1).cpu())
    return torch.cat(embs)

IMG_EMB = clip_image_emb(IDS)

def clip_scores(ids, captions):
    T = clip_text_emb([str(c) for c in captions])
    I = torch.stack([IMG_EMB[i] for i in ids])
    return (2.5 * (I * T).sum(-1).clamp(min=0)).numpy()

### Vérification des métriques sur les légendes humaines

In [ ]:
# Validation sur les légendes humaines : une légende humaine contre les 4 autres
human_pred = {k: REFS[k][0] for k in IDS}
human_row = text_metrics(human_pred, {k: REFS[k][1:] for k in IDS})
human_row["CLIPScore"] = float(clip_scores(IDS, [human_pred[k] for k in IDS]).mean())
human_row["mots/légende"] = float(np.mean([len(human_pred[k].split()) for k in IDS]))

# Contrôle : une légende d'une AUTRE image doit avoir des scores nettement plus bas
shuffled = IDS[1:] + IDS[:1]
wrong_pred = {k: REFS[s][0] for k, s in zip(IDS, shuffled)}
wrong_row = text_metrics(wrong_pred, {k: REFS[k][1:] for k in IDS})
wrong_row["CLIPScore"] = float(clip_scores(IDS, [wrong_pred[k] for k in IDS]).mean())

check = pd.DataFrame([{"cas": "légende humaine correcte", **human_row}, {"cas": "légende d'une autre image", **wrong_row}])
check.to_csv(RES_DIR / "validation_metriques.csv", index=False)
check.round(3)

## Scores des légendes générées
Utilise les légendes produites par Wafae (dossier partagé). Les configurations pas encore générées sont signalées.

In [ ]:
# Charge les légendes disponibles (produites par Wafae) et calcule toutes les métriques
CAPS, rows = {}, []
for name in MODELS:
    for prec in PRECISIONS:
        tag = f"{name}__{prec}"
        cap_path, stat_path = RES_DIR / f"captions_{tag}.csv", RES_DIR / f"stats_{tag}.json"
        if not cap_path.exists():
            print(f"[manquant] {tag} — pas encore généré par Wafae"); continue
        df = pd.read_csv(cap_path).set_index("id")
        ids = [i for i in IDS if i in df.index]
        df = df.loc[ids]; df["caption"] = df["caption"].fillna("").astype(str)
        CAPS[(name, prec)] = df["caption"].to_dict()
        stats = json.loads(stat_path.read_text()) if stat_path.exists() else {"model": name, "precision": prec}
        m = text_metrics(CAPS[(name, prec)], REFS)
        m["CLIPScore"] = float(clip_scores(ids, df["caption"].tolist()).mean())
        m["mots/légende"] = float(df["caption"].str.split().str.len().mean())
        if prec != "16bit" and (name, "16bit") in CAPS:
            base = CAPS[(name, "16bit")]
            common_ids = [k for k in ids if k in base]
            fid = text_metrics({k: CAPS[(name, prec)][k] for k in common_ids}, {k: [base[k]] for k in common_ids})
            m["BLEU-4 vs 16bit"] = fid["BLEU-4"]
            m["% identique au 16bit"] = 100 * np.mean([tok(CAPS[(name, prec)][k]) == tok(base[k]) for k in common_ids])
        rows.append({**stats, **m})

rows.append({"model": "Humain (réf. 0 vs 4 autres)", "precision": "-", **human_row})
results = pd.DataFrame(rows)
results.to_csv(RES_DIR / "resultats_finaux.csv", index=False)
show = [c for c in ["model", "precision", "weights_GB", "peak_GPU_GB", "time_per_image_s", "BLEU-1", "BLEU-4",
                    "ROUGE-L", "CIDEr", "CLIPScore", "mots/légende", "BLEU-4 vs 16bit", "% identique au 16bit"]
        if c in results.columns]
results[show].round(3)

## Graphiques et exemples

In [ ]:
# Graphiques de comparaison
res = results[results["precision"] != "-"].copy()
models = list(MODELS)
colors = {"16bit": "#2a6fdb", "8bit": "#f2a541", "4bit": "#3bb273"}
metrics = [("weights_GB", "Mémoire des poids (Go)"), ("peak_GPU_GB", "Pic mémoire GPU (Go)"),
           ("time_per_image_s", "Temps par image (s)"), ("CIDEr", "CIDEr (↑)"),
           ("CLIPScore", "CLIPScore (↑)"), ("BLEU-4", "BLEU-4 (↑)")]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
x = np.arange(len(models)); w = 0.26
for ax, (col, title) in zip(axes.ravel(), metrics):
    for j, prec in enumerate(PRECISIONS):
        vals = [res[(res.model == m) & (res.precision == prec)][col].values[0] for m in models]
        bars = ax.bar(x + (j - 1) * w, vals, w, label=prec, color=colors[prec])
        ax.bar_label(bars, fmt="%.2f", fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(models); ax.set_title(title)
    ax.spines[["top", "right"]].set_visible(False)
axes[0, 0].legend(title="Précision")
plt.tight_layout(); plt.savefig(RES_DIR / "comparaison.png", dpi=150); plt.show()

# Compromis mémoire / qualité
fig, ax = plt.subplots(figsize=(7, 5))
for m, marker in zip(models, ["o", "s"]):
    for prec in PRECISIONS:
        r = res[(res.model == m) & (res.precision == prec)].iloc[0]
        ax.scatter(r.weights_GB, r.CIDEr, s=120, marker=marker, color=colors[prec])
        ax.annotate(f"{m}\n{prec}", (r.weights_GB, r.CIDEr), fontsize=8, xytext=(6, 4), textcoords="offset points")
ax.set_xlabel("Mémoire des poids (Go)"); ax.set_ylabel("CIDEr")
ax.set_title("Compromis mémoire / qualité"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(RES_DIR / "compromis_memoire_qualite.png", dpi=150); plt.show()

In [ ]:
# Exemples qualitatifs : mêmes images, 6 configurations
for img_id in IDS[:4]:
    plt.figure(figsize=(4, 4)); plt.imshow(load_img(img_id)); plt.axis("off"); plt.show()
    print("Référence humaine :", REFS[img_id][0])
    for (name, prec), caps in CAPS.items():
        print(f"  {name:12s} {prec:6s} → {caps[img_id]}")
    print("-" * 100)

### À analyser
- Quel modèle légende le mieux ? Lequel supporte le mieux la quantification ?
- Les pertes en 8 et 4 bits sont-elles importantes ?
- Pourquoi BLEU et CIDEr restent bas même pour de bonnes légendes ?

## 6. Option A — Jeu de données de légendes pour entraîner une LoRA

On exporte les images + légendes de la meilleure configuration (ou d'une configuration choisie) au format utilisé par les scripts d'entraînement LoRA :
- `metadata.jsonl` (format `imagefolder` de 🤗 *diffusers*) ;
- un fichier `.txt` par image (format *kohya_ss*).

Un **mot déclencheur** optionnel (`TRIGGER_WORD`) est ajouté devant chaque légende pour pouvoir activer le style de la LoRA.

In [ ]:
# Choix de la configuration : par défaut, la meilleure en CIDEr. Tu peux forcer ex. ("Qwen2-VL-2B", "4bit").
best = res.sort_values("CIDEr", ascending=False).iloc[0]
LORA_CONFIG = (best.model, best.precision)
TRIGGER_WORD = ""          # ex. "flkr style" ; vide = pas de mot déclencheur
print("Configuration utilisée pour la LoRA :", LORA_CONFIG)

LORA_DIR = ROOT / "lora_dataset" / "train"
if LORA_DIR.exists():
    shutil.rmtree(LORA_DIR)
LORA_DIR.mkdir(parents=True)

with open(LORA_DIR / "metadata.jsonl", "w", encoding="utf-8") as f:
    for img_id in IDS:
        cap = CAPS[LORA_CONFIG][img_id]
        cap = f"{TRIGGER_WORD}, {cap}" if TRIGGER_WORD else cap
        shutil.copy(IMG_DIR / f"{img_id}.jpg", LORA_DIR / f"{img_id}.jpg")
        (LORA_DIR / f"{img_id}.txt").write_text(cap, encoding="utf-8")
        f.write(json.dumps({"file_name": f"{img_id}.jpg", "text": cap}, ensure_ascii=False) + "\n")

shutil.make_archive(str(ROOT / "lora_dataset"), "zip", ROOT / "lora_dataset")
print(f"{len(IDS)} paires image/légende exportées → {ROOT / 'lora_dataset.zip'}")
print(open(LORA_DIR / "metadata.jsonl").readlines()[:3])

### (Optionnel) Entraîner une petite LoRA Stable Diffusion 1.5 avec ces légendes

Mets `RUN_LORA_TRAINING = True` pour lancer un **petit** entraînement (≈ 30–45 min sur T4 pour 500 pas). Le sujet n'exige pas forcément cet entraînement : le livrable principal de l'option A est le jeu de légendes ci-dessus.

In [ ]:
RUN_LORA_TRAINING = False

if RUN_LORA_TRAINING:
    import subprocess
    gc.collect(); torch.cuda.empty_cache()
    subprocess.run("git clone --depth 1 https://github.com/huggingface/diffusers /content/diffusers", shell=True, check=False)
    subprocess.run("pip install -q /content/diffusers peft ftfy tensorboard", shell=True, check=True)
    cmd = f'''accelerate launch --mixed_precision=fp16 /content/diffusers/examples/text_to_image/train_text_to_image_lora.py \
      --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
      --train_data_dir="{LORA_DIR}" --caption_column="text" --image_column="image" \
      --resolution=512 --center_crop --random_flip \
      --train_batch_size=1 --gradient_accumulation_steps=4 --gradient_checkpointing \
      --max_train_steps=500 --learning_rate=1e-4 --lr_scheduler="cosine" --lr_warmup_steps=0 \
      --rank=4 --seed={SEED} --output_dir="{ROOT / 'lora_sd15'}"'''
    subprocess.run(cmd, shell=True, check=True)

    from diffusers import StableDiffusionPipeline
    pipe = StableDiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5",
                                                   torch_dtype=torch.float16).to("cuda")
    pipe.load_lora_weights(str(ROOT / "lora_sd15"))
    prompt = (TRIGGER_WORD + ", " if TRIGGER_WORD else "") + "a dog running on the beach"
    image = pipe(prompt, num_inference_steps=30, generator=torch.Generator("cuda").manual_seed(SEED)).images[0]
    image.save(ROOT / "exemple_lora.png"); display(image)
    del pipe; gc.collect(); torch.cuda.empty_cache()

## Mettre à jour GitHub
1. *Fichier → Enregistrer une copie dans GitHub* : dépôt `vlm-compacts`, branche **ta branche** (wafae, meriem ou fatima), chemin **`notebooks/02_evaluation_lora.ipynb`**, avec un message qui dit ce qui a été fait, par exemple `Ajout des graphiques de comparaison`.
2. Exécute la cellule ci-dessous, décompresse le zip et ajoute les fichiers dans le dossier `results/` de ta branche, puis ouvre une pull request vers `main` (*Add file → Upload files*).

In [ ]:
# Rassemble les résultats légers (CSV, JSON, PNG) à pousser — jamais les images ni les modèles
import zipfile, glob
zip_path = ROOT / "results_meriem.zip"
files = []
for pat in ['validation_metriques.csv', 'resultats_finaux.csv', 'comparaison.png', 'compromis_memoire_qualite.png']:
    files += glob.glob(str(RES_DIR / pat))
with zipfile.ZipFile(zip_path, "w") as z:
    for f in sorted(set(files)):
        z.write(f, arcname=Path(f).name)
print(len(set(files)), "fichiers →", zip_path)
try:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))
except Exception:
    pass